# Rama PEFT (LoRA) — Consigna 9b

Este notebook implementa la estrategia **PEFT con LoRA** sobre un modelo pequeño de pesos abiertos (GPT-2, 124M de parámetros), pensado para correr en **Google Colab con GPU T4 gratuita**.

**Antes de ejecutar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4).

**Qué tenés que hacer vos:** reemplazar el dataset de ejemplo (celda marcada con `TODO`) por ejemplos propios de tu caso de uso (consigna 1), y ajustar los 2-3 hiperparámetros marcados con `TODO`. El resto del notebook ya está resuelto y corre de punta a punta.

## 1. Verificar GPU disponible

In [ ]:
!nvidia-smi

## 2. Instalación de dependencias

In [ ]:
!pip install -q transformers==4.44.2 peft==0.13.0 datasets==2.21.0 accelerate==0.34.2

## 3. Constantes del experimento

Todo valor "mágico" va acá arriba, nombrado, para no repetirlo suelto en el resto del notebook.

In [ ]:
NOMBRE_MODELO_BASE = "gpt2"  # 124M de parámetros, sin gating de Hugging Face

# Hiperparámetros de LoRA. TODO: podés ajustar r y lora_alpha si querés experimentar.
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["c_attn"]  # capa de atención combinada (q, k, v) en GPT-2

# Hiperparámetros de entrenamiento. TODO: podés ajustar EPOCHS si tu dataset es más grande.
EPOCHS = 15
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
LONGITUD_MAXIMA_TOKENS = 64

CARPETA_SALIDA = "./resultado_lora"

## 4. Cargar modelo base y tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(NOMBRE_MODELO_BASE)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 no trae un pad_token propio

modelo_base = AutoModelForCausalLM.from_pretrained(NOMBRE_MODELO_BASE)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo_base.to(dispositivo)
print(f"Modelo cargado en: {dispositivo}")

## 5. Generación ANTES del fine-tuning

Guardamos estas respuestas para compararlas con las del modelo ajustado al final del notebook.

In [ ]:
def generar_texto(modelo, prompt: str) -> str:
    """Genera una continuación de texto para un prompt dado con el modelo recibido."""
    entrada = tokenizer(prompt, return_tensors="pt").to(dispositivo)
    salida = modelo.generate(
        **entrada,
        max_new_tokens=LONGITUD_MAXIMA_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(salida[0], skip_special_tokens=True)


# TODO: reemplazá estos prompts por preguntas típicas de tu propio caso de uso.
PROMPTS_DE_PRUEBA = [
    "Consulta: ¿Cómo cancelo una suscripción activa?\nRespuesta:",
    "Consulta: ¿Qué medios de pago aceptan?\nRespuesta:",
]

respuestas_antes = [generar_texto(modelo_base, prompt) for prompt in PROMPTS_DE_PRUEBA]
for prompt, respuesta in zip(PROMPTS_DE_PRUEBA, respuestas_antes):
    print(f"--- {prompt}\n{respuesta}\n")

## 6. Dataset de ejemplos propios (TODO: reemplazar)

Reemplazá esta lista por ejemplos reales de tu caso de uso (consigna 1): pares de consulta/respuesta que reflejen cómo querés que responda el modelo ajustado.

In [ ]:
# TODO: reemplazar por ejemplos propios del caso de uso (mínimo recomendado: 10-20 ejemplos).
EJEMPLOS_PROPIOS = [
    {"consulta": "¿Cómo cancelo una suscripción activa?",
     "respuesta": "Andá a Configuración > Suscripción > Cancelar y confirmá la baja."},
    {"consulta": "¿Qué medios de pago aceptan?",
     "respuesta": "Aceptamos tarjeta de crédito, débito y transferencia bancaria."},
    {"consulta": "¿Puedo cambiar mi plan en cualquier momento?",
     "respuesta": "Sí, podés cambiar de plan cuando quieras desde tu panel de usuario."},
]

In [ ]:
from datasets import Dataset


def formatear_ejemplo(ejemplo: dict) -> dict:
    """Convierte un par consulta/respuesta al mismo formato de prompt usado en main.py."""
    texto = f"Consulta: {ejemplo['consulta']}\nRespuesta: {ejemplo['respuesta']}{tokenizer.eos_token}"
    return {"texto": texto}


def tokenizar(ejemplo: dict) -> dict:
    return tokenizer(
        ejemplo["texto"],
        truncation=True,
        max_length=LONGITUD_MAXIMA_TOKENS,
        padding="max_length",
    )


dataset = Dataset.from_list(EJEMPLOS_PROPIOS)
dataset = dataset.map(formatear_ejemplo)
dataset = dataset.map(tokenizar, batched=True, remove_columns=dataset.column_names)
dataset = dataset.map(lambda ejemplo: {"labels": ejemplo["input_ids"]})

## 7. Configurar LoRA y aplicarlo al modelo

In [ ]:
from peft import LoraConfig, get_peft_model

configuracion_lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
)

modelo_lora = get_peft_model(modelo_base, configuracion_lora)
modelo_lora.print_trainable_parameters()

## 8. Entrenamiento

In [ ]:
from transformers import Trainer, TrainingArguments

argumentos_entrenamiento = TrainingArguments(
    output_dir=CARPETA_SALIDA,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

entrenador = Trainer(
    model=modelo_lora,
    args=argumentos_entrenamiento,
    train_dataset=dataset,
)

entrenador.train()

## 9. Comparación ANTES / DESPUÉS del ajuste

In [ ]:
modelo_lora.eval()
respuestas_despues = [generar_texto(modelo_lora, prompt) for prompt in PROMPTS_DE_PRUEBA]

for prompt, antes, despues in zip(PROMPTS_DE_PRUEBA, respuestas_antes, respuestas_despues):
    print(f"=== {prompt}")
    print(f"ANTES:    {antes}")
    print(f"DESPUÉS:  {despues}")
    print()

## 10. Guardar los adaptadores LoRA (opcional)

Descargá esta carpeta si querés adjuntar los adaptadores entrenados como evidencia adicional (no es obligatorio para la entrega).

In [ ]:
modelo_lora.save_pretrained(CARPETA_SALIDA)
print(f"Adaptadores LoRA guardados en {CARPETA_SALIDA}")